# Business Intelligence Report: Hotel Booking Optimization

### Project Overview
This analysis explores booking patterns, cancellation dynamics, and revenue streams for a hotel dataset. The primary goal is to extract actionable insights that can help management reduce revenue leakage and identify high-value customer segments.

### Executive Objectives:
* **Financial Impact:** Quantify the "Opportunity Cost" of cancellations.
* **Operational Efficiency:** Analyze how booking lead times correlate with cancellation risks.
* **Market Valuation:** Identify the top-performing geographic markets based on Average Daily Rate (ADR).

---

### 1. Revenue & Cancellation Analysis
In this section, we calculate the total volume of bookings and the **Estimated Revenue** (defined as `ADR * total_nights`). By segmenting these by hotel type and cancellation status, we can visualize the financial impact of lost bookings.

In [8]:
import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect('../data/hotel_database.db')

query_revenue = """
SELECT 
    hotel,
    is_canceled,
    COUNT(*) as total_bookings,
    ROUND(SUM(adr * total_nights), 2) as estimated_revenue
FROM bookings_analytics
GROUP BY 1, 2;
"""

df_rev = pd.read_sql(query_revenue, conn)
df_rev['status'] = df_rev['is_canceled'].map({0: 'Confirmed', 1: 'Canceled'})

display(df_rev.head())

,hotel,is_canceled,total_bookings,estimated_revenue,status
0,City Hotel,0,45119,14385342.00,Confirmed
1,City Hotel,1,32973,10885059.78,Canceled
2,Resort Hotel,0,28269,11601634.03,Confirmed
3,Resort Hotel,1,11038,5842177.34,Canceled


In [11]:
fig = px.bar(df_rev, 
             x='hotel', 
             y='estimated_revenue', 
             color='status',
             barmode='group',
             title='Revenue from Canceled vs Confirmed Bookings by Hotel Type',
             labels={'estimated_revenue': 'Estimated Revenue (€)', 'hotel': 'Hotel Type'},
             text='estimated_revenue', 
             template='plotly_white')

fig.update_traces(texttemplate='%{text:.2s}', textposition='outside') 
fig.show()

#### Key Findings: Revenue Impact
* **Revenue Leakage:** The *City Hotel* is experiencing a significantly higher volume of canceled revenue (exceeding €10M) compared to the *Resort Hotel*. 
* **Cancellation Ratio:** While both hotels suffer from cancellations, the *City Hotel* has a tighter gap between confirmed and canceled revenue, suggesting a more volatile booking ecosystem.
* **Business Opportunity:** Reducing the cancellation rate in the *City Hotel* by even 10% could recover millions in potentially lost revenue. 

---

### 2. Booking Lead Time & Cancellation Risk
In this section, we analyze the correlation between how far in advance a guest books (**Lead Time**) and the likelihood of them canceling. 

We have categorized the lead time into four strategic windows:
* **Last Minute:** < 7 days
* **Short Term:** 7 - 30 days
* **Medium Term:** 31 - 90 days
* **Long Term:** > 90 days

Understanding this relationship is crucial for implementing tiered cancellation policies and managing overbooking strategies.

In [10]:
query_lead_time = """
SELECT 
    CASE 
        WHEN lead_time < 7 THEN 'Last Minute'
        WHEN lead_time BETWEEN 7 AND 30 THEN 'Short Term'
        WHEN lead_time BETWEEN 31 AND 90 THEN 'Medium Term'
        ELSE 'Long Term'
    END as booking_window,
    AVG(is_canceled) * 100 as cancellation_rate
FROM bookings_analytics
GROUP BY 1
ORDER BY cancellation_rate DESC;
"""

df_lt = pd.read_sql(query_lead_time, conn)
display(df_lt)

,booking_window,cancellation_rate
0,Long Term,50.824894
1,Medium Term,37.934801
2,Short Term,27.141926
3,Last Minute,9.461334


In [12]:
import plotly.express as px

category_order = ['Last Minute', 'Short Term', 'Medium Term', 'Long Term']

fig_lt = px.bar(df_lt, 
                x='booking_window', 
                y='cancellation_rate',
                title='Probability of Cancellation by Booking Window',
                labels={'booking_window': 'Booking Window', 'cancellation_rate': 'Cancellation Rate (%)'},
                category_orders={'booking_window': category_order},
                color='cancellation_rate',
                color_continuous_scale='Reds',
                template='plotly_white')

fig_lt.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
fig_lt.show()

#### Key Findings: The "Long Term" Risk Factor
* **Critical Cancellation Threshold:** There is a drastic escalation in risk as the booking window expands. **Long Term bookings (>90 days)** have a staggering **50.8% cancellation rate**, making them the most unstable segment for the hotel's inventory.
* **The "Safety" Zone:** **Last Minute** bookings are highly reliable, with only a **9.5%** cancellation rate. This suggests that guests booking close to their arrival date have a much higher "intent to stay."

---

### 3. High-Value Market Analysis (ADR by Country)
In this final section, we identify the geographic markets that generate the highest **Average Daily Rate (ADR)**. To ensure statistical relevance, we only include countries with more than 100 confirmed bookings.

This analysis helps the Marketing and Sales departments understand which nationalities are willing to pay a premium for their stay, allowing for more targeted high-value advertising campaigns.

In [13]:
query_countries = """
SELECT 
    country,
    COUNT(*) as total_bookings,
    ROUND(AVG(adr), 2) as avg_daily_rate
FROM bookings_analytics
WHERE is_canceled = 0
GROUP BY country
HAVING total_bookings > 100
ORDER BY avg_daily_rate DESC
LIMIT 5;
"""

df_countries = pd.read_sql(query_countries, conn)
display(df_countries)

,country,total_bookings,avg_daily_rate
0,LUX,176,130.77
1,JPN,169,120.52
2,USA,1585,119.80
3,MAR,147,119.04
4,CHE,1293,117.68


In [7]:
fig_countries = px.bar(df_countries, 
                       x='avg_daily_rate', 
                       y='country',
                       orientation='h',
                       title='Top 5 Mercados por Gasto Diario Medio (ADR)',
                       labels={'avg_daily_rate': 'Precio Medio por Noche (€)', 'country': 'País'},
                       text='avg_daily_rate',
                       color='avg_daily_rate',
                       color_continuous_scale='Viridis',
                       template='plotly_white')

fig_countries.update_layout(yaxis={'categoryorder':'total ascending'})
fig_countries.show()

#### Key Findings: Geographic Pricing Power
* **Top-Tier Markets:** The identified Top 5 countries represent the "Premium" segment of the hotel's clientele. These guests accept a higher Average Daily Rate, significantly boosting the RevPAR (Revenue Per Available Room).
* **Quality over Quantity:** While some countries might provide a high volume of guests, these 5 specific markets are the most profitable per room/night, justifying dedicated luxury services or localized marketing efforts.
* **Confirmed Success:** By filtering for `is_canceled = 0`, we are looking at real, realized revenue, not just intent. These are proven high-value markets.

---

## Executive Summary & Final Conclusions

After an end-to-end analysis of the hotel booking data, we have reached three transformative conclusions for the business:

1. **Revenue Leakage:** The City Hotel faces a massive financial challenge with cancellations. 
2. **The 50% Rule:** Bookings made more than 90 days in advance are essentially a "coin flip" (50.8% cancellation). 
3. **Market Focus:** Focusing on the Top 5 ADR countries will allow the hotel to increase total revenue without necessarily increasing occupancy, by attracting guests who value the service at a higher price point.

**Project Status: Analysis Complete. Ready for Deployment of Business Recommendations.**